A dictionary is defined with a list of 'expectations' which will be applied as a 'expect_all_or_fail' on the dp table 

In [0]:
valid_rows = {
    "not_null_pii": "PII IS NOT NULL",
    "valid_date": "date IS NOT NULL"
}

@dp.table(
    comment="This table will be used to ingest the raw CSV files and add metadata columns to the bronze table.",
    table_properties={"quality": "bronze"}
)
@dp.expect_all_or_fail(valid_rows)
def health_bronze():
    return (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(project_functions.get_health_csv_schema())
        .load(raw_data_path)
        .select(
            "*",
            "_metadata.file_name",
            "_metadata.file_modification_time",
            F.current_timestamp().alias("processing_time")
        )
    )

Another method - declaring a dictionary with expected values of a test scenario

In [0]:
target_integration_tests_validation = {
    'development': {
        'health_bronze': {'total_rows': 7500},
        'health_silver': {'total_rows': 7500}
    },
    'stage': {
        'health_bronze': {'total_rows': 35000},
        'health_silver': {'total_rows': 35000}
    }
}
if target in ('development', 'stage'):
    total_expected_bronze = target_integration_tests_validation[target]['health_bronze']['total_rows']
    total_expected_silver = target_integration_tests_validation[target]['health_silver']['total_rows']

In [0]:
def test_count_table_total_rows(table_name, total_count, target):
    @dp.table(
        name=f"TEST_{target}_{table_name}_total_rows_verification",
        comment=f"Confirms all rows were ingested from the {target} raw data to {table_name}"
    )
    @dp.expect_all_or_fail({"valid count": f"total_rows = {total_count}"})
    def count_table_total_rows():
        return spark.sql(f"""
            SELECT COUNT(*) AS total_rows FROM LIVE.{table_name}
        """)

`target` is passed as a config value from pipeline setting and read using spark.conf.get

In [0]:
if target in ('development', 'stage'):
    test_count_table_total_rows('health_bronze', total_expected_bronze, target)
    test_count_table_total_rows('health_silver', total_expected_silver, target)
    test_gold_table_columns()
elif target == 'production':
    test_gold_table_columns()

Another dictionary with,  SQL snippets to check during validation(expectation)

In [0]:
check_silver_calc_columns = {
    "valid age group": "Age_Group in ('0-9', '10-19', '20-29', '30-39', '40-49', '50+', 'Unknown')",
    "valid cholest group": "HighCholest_Group in ('Normal', 'Above Average', 'High', 'Unknown')"
}

perform the test using expectation - `expect_all_or_fail`

In [0]:
@dp.table(comment="Check age group and high cholest group in the gold table")
@dp.expect_all_or_fail(check_silver_calc_columns)
def test_calculated_columns_age_cholesterol():
    return (dp
            .read("chol_age_agg")
            .select("Age_Group", "HighCholest_Group")
    )

In [0]:
@dp.table(
    comment="This table will create, drop and categorize columns from the bronze table.",
    table_properties={"quality": "bronze"}
)
def health_silver():
    return (
        dp
        .read_stream("health_bronze")
        .withColumn("HighCholest_Group", project_functions.high_cholest_map("HighCholest"))  # UDF - highcholest_map
        .withColumn("Age_Group", project_functions.group_ages_map("Age"))                     # UDF - group_ages_map Age
        .drop("file_name", "file_modification_time", "processing_time")                      # Drop unnecessary metadata columns
    )